<a href="https://colab.research.google.com/github/luhipi/sarvey-tutorials/blob/main/notebooks/workshops/ISPRS26/SARvey_tutorial_ISPRS26_Toronto.ipynb
" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<p>
  <img src="https://www.uni-hannover.de/typo3conf/ext/luh_website/Resources/Public/Images/Logo/luh_logo.svg" width="128" />
  <img src="https://www.ipi.uni-hannover.de/fileadmin/site-templates/logos/ipi/ipi_logo.png" width="38" />
  <img src="https://static.wixstatic.com/media/4a7c8c_55e4ee8611e6478cbf05ded9115e3299~mv2.png/v1/fill/w_234,h_50,al_c,q_85,usm_0.66_1.00_0.01,enc_avif,quality_auto/ISPRS%202026_Horizontal%20Logo%20-%20Colour.png" width="210" />
</p>

#### InSAR Time Series Analysis with SARvey and InSAR Explorer

##### Tutorial @ [ISPRS 2026, July 2026, Toronto, Canada](https://www.isprs2026toronto.com)

##### **Everything stable in Toronto?**


---
<img src="https://seafile.projekt.uni-hannover.de/f/69eac9bb5f86400fa1ad/?dl=1" width="512">


**SARvey** is an open-source software developed for the analysis of Interferometric Synthetic Aperture Radar (InSAR) displacement time series, particularly tailored for engineering applications. It offers a comprehensive workflow to process and analyze Synthetic Aperture Radar (SAR) data, enabling users to monitor and assess ground deformations with high precision.

For more information, visit the following:
* **[SARvey GitHub Repository](https://github.com/luhipi/sarvey)**
* **[SARvey Documentation](https://sarvey.readthedocs.io/)**
* **[How to cite SARvey](https://sarvey.readthedocs.io/main/readme.html#how-to-cite/)**
* **[SARvey peer-reviewed paper](https://www.sciencedirect.com/science/article/pii/S1364815226001519)**

---

This tutorial is developed based on **[SARvey v1.3.0](https://github.com/luhipi/sarvey/tree/v1.3.0)** tag and covers the processing steps for a sample dataset, as a practical example to demonstrate its main capabilities.

---

*This tutorial is prepared by **Andreas Piter** from the [Institute of Photogrammetry and GeoInformation](https://www.ipi.uni-hannover.de/en/), Leibniz University Hannover.*



## Installation

In [ ]:
! pip install git+https://github.com/luhipi/sarvey.git@v1.3.0 --quiet

In [ ]:
! sarvey -h

In [ ]:
! apt-get -qq install tree

## Imports

In [ ]:
import os
from IPython.display import display, Image, JSON, Markdown
from matplotlib import pyplot as plt
import numpy as np
import h5py as h5
import json5 as json

## Download data

**Background**

<img src="https://seafile.projekt.uni-hannover.de/f/5fdb043e3a3f43bc8b0f/?dl=1" width="512">


---

This study area covers the venue of the ISPRS congress 2026. Urban areas can cause multiple challenges due to geometrical distortions occuring in SAR images (see figure).

<img src="https://seafile.projekt.uni-hannover.de/f/2b9260c7fdc14402b4e9/?dl=1" width="512">

L: Layover
S: Shadow

---

Todo: add description

<img src="https://seafile.projekt.uni-hannover.de/f/18fd268d86b54846bdda/?dl=1" width="1200">


**Demo Dataset**: Toronto

**Dataset Highlights:**
- **Location:** Toronto, Canada
- **Sensor:** Sentinel-1
- **Number of images:** 270
- **Temporal interval:** 15.03.2015-16.05.2026
- **Data Type:** Coregistered stack of SLCs with corresponding geometry information

---

In [ ]:
# specify working directory. On Google Colab it should be '/content'
work_dir = '/content'

In [ ]:
# Change the directory
os.chdir(work_dir)

# Download data
! wget  https://seafile.projekt.uni-hannover.de/f/4ce77b3309bb4e0c96c4/?dl=1  -O Toronto_S1_asc_2016_2026.zip --show-progress -q

# Unzip data into toronto directory
! unzip -q -o Toronto_S1_asc_2016_2026.zip

# Rename the extracted folder
! mv Toronto_S1_asc_2016_2026 toronto_s1

# Remove the zip file
! rm Toronto_S1_asc_2016_2026.zip

# Define the project directory path as a variable
project_dir=os.path.join(work_dir, 'toronto_s1')

In [ ]:
# Navigate to the project directory
os.chdir(project_dir)

# Display the directory structure in a tree-like format
! tree

# SARvey processing

We process using a small baseline interferogram network and analyse the estimated atmospheric effect.

Let's create a config file.

In [ ]:
os.chdir(project_dir)

# Todo: Create a configuration file called config.json
! sarvey -f config.json -g

We define the function to read and write the configuration parameters programmatically.

In [ ]:
def loadJsonConfig(config_file):
    # Load the contents of config_file into a Python dictionary
    with open('config.json', 'r') as f:
        config_dict = json.load(f)
    return config_dict

def dumpToJsonConfig(config_file, config_dict):
    # Write the config_dict to config_file
    with open(config_file, 'w') as f:
        json.dump(config_dict, f, indent=4)

Now, we want to display the content of the configuration file to look for the parameters we want to change.

In [ ]:
os.chdir(project_dir)

# Load the configs
config = loadJsonConfig('config.json')

# Display the configs
config


We want to change a few parameters for our processing.

1.   Use 10 cores for the parallel processing (already specified)
2.   Reduce the grid size for the first-order points to 100m.
3.   Select only 10 nearest neighbours for connecting the first-order points.


In [ ]:
os.chdir(project_dir)

# Load the configs
config = loadJsonConfig('config.json')

# Modify parameter
config['general']['num_cores'] = 10
config['consistency_check']['grid_size'] = 100
config['consistency_check']['num_nearest_neighbours'] = 10

dumpToJsonConfig('config.json', config)

## Step 0: Preparation

In [ ]:
os.chdir(project_dir)

# Todo: Process step 0 with SARvey.

! sarvey -f config.json 0 0

After the processing, check the results to answer these questions:

1.   How does the mean amplitude image look like?
2.   Are there any particularities in the interferogram network?
3.   What is the general coherence level in the study area? Are the pixels coherent in your area of interest?

In [ ]:
! tree

In [ ]:
Image(filename='outputs/pic/step_0_interferogram_network.png')

In [ ]:
# Display the output images
Image(filename='outputs/pic/step_0_amplitude_image.png')

In [ ]:
Image(filename='outputs/pic/step_0_temporal_phase_coherence.png')

Quickly note your main findings here:

1.  ...
2.  ...
3.  ...

## Step 1: Consistency

In [ ]:
os.chdir(project_dir)

# Todo: Process step 1 with SARvey.
! sarvey -f config.json 1 1

In [ ]:
# Display initial first order network
Image(filename='outputs/pic/step_1_network_0_initial.png')

In [ ]:
# Display first order network after outlier removal
Image(filename='outputs/pic/step_1_network_1_points_removed.png')

In [ ]:
# Display first order network after retriangulation
Image(filename='outputs/pic/step_1_network_1_points_retriangulated.png')

In [ ]:
# Display final first order network
Image(filename='outputs/pic/step_1_network_2_arcs_removed.png')

## Step 2: Unwrapping

In [ ]:
os.chdir(project_dir)

# todo: Process step 2 with SARvey.
! sarvey -f config.json 2 2

In [ ]:
# Display Estimated Dem Correction
Image(filename='outputs/pic/step_2_estimation_dem_correction.png')

In [ ]:
# Display Estimated Velocity
Image(filename='outputs/pic/step_2_estimation_velocity.png')

## Step 3: Filtering

In [ ]:
os.chdir(project_dir)

# todo: Process step 3 with SARvey.
! sarvey -f config.json 3 3

The atmosphere is filtered based on a set of good pixels which should be well distributed over the study area. We use a grid to choose the most suitable pixels in a (more or less) regular sampling. In the config, the grid size is specified.

❓ Look at the selection of the coherent points. Do you think the selection is good?

In [ ]:
# Display the stable points used for filtering
Image(filename='outputs/pic/step_3_stable_points.png')

Let's check the current grid size in the config file which is given in meters.

In [ ]:
os.chdir(project_dir)

# Load the configs
config = loadJsonConfig('config.json')
config['filtering']['grid_size']

Let's change the grid size to 300m and see how the selection of the stable points changes.

In [ ]:
os.chdir(project_dir)

# Load the configs
config = loadJsonConfig('config.json')

# Modify parameter
config['filtering'][''] = 300

dumpToJsonConfig('config.json', config)

Let's rerun the filtering step with the new grid size.

In [ ]:
os.chdir(project_dir)

# todo: Process step 3 with SARvey.
! sarvey -f config.json 3 3

In [ ]:
# Display the stable points used for filtering
Image(filename='outputs/pic/step_3_stable_points.png')

Now, we want to analyse the estimated atmospheric effect. For this, we focus on the pixels which were used during filtering. The atmospheric phase for each image acquisition for the first-order points is stored in p1_aps.h5. Let's plot the estimated atmospheric effect for the first-order points.

In [ ]:
# create the figure which is stored in a file
! sarvey_plot outputs/p1_aps.h5 -p

# Display the atmospheric effect for the first-order points
Image(filename='outputs/pic/p1/p1_aps_phase.png')

Each subfigure shows the estimated atmospheric effect for one image acquisition at the first-order points.

❓ How does the atmospheric effect look like? Is it the same in all images? Discuss your findings with your neighbours and the tutorial instructurs.

Quickly note your main findings here:

*  ...
*  ...
*  ...

## Step 4: Densification

In [ ]:
os.chdir(project_dir)

# todo: Process step 4 with SARvey.
! sarvey -f config.json 4 4

## Export to InSAR Explorer

In [ ]:
os.chdir(project_dir)

! sarvey_export outputs/p2_coh80_ts.h5 -g -o outputs/toronto_coh80.gpkg

❓ Export and check the results in QGIS using the InSAR Explorer. Do the results match the expected displacement pattern? Which displacement patterns do you find? Are they plausible? Discuss your findings with your neighbours and the tutorial instructurs.